#Bronze to Silver
This notebook refines raw data from the Bronze layer by applying additional validations, transformations, and Change Data Capture (CDC) logic. The Silver layer contains enriched and validated data ready for analytics.


## Step 3: Validating and Transforming Orders Data

- Filter and enrich orders data with quality constraints and transformations.

- **Validations:**
  - **valid_date:** Ensures order_timestamp is later than January 1, 2021.
- **Transformations:**
    - Converts order_timestamp to a timestamp format.
    - Excludes unnecessary fields like _rescued_data.


In [0]:
CREATE OR REFRESH STREAMING TABLE order_table_silver
  (CONSTRAINT valid_date EXPECT (order_timestamp > "2021-01-01") ON VIOLATION FAIL UPDATE)
COMMENT "Append only orders with valid timestamps"
TBLPROPERTIES ("quality" = "silver")
AS 
SELECT 
  timestamp(order_timestamp) AS order_timestamp, 
  * EXCEPT (order_timestamp, _rescued_data)
FROM STREAM(LIVE.order_table_bronze)                    -- References the orders_bronze streaming table

## Step 4: Processing Customers CDC Data with APPLY CHANGES INTO

- Process customer CDC data into a Type 1 Slowly Changing Dimension (SCD) table.
- **CDC Logic:**
  - Applies inserts, updates, and deletes to maintain the latest state.
  - Uses `customer_id` as the primary key.
  - Orders operations by the `timestamp` field.

In [0]:
CREATE OR REFRESH STREAMING TABLE customers_silver;

APPLY CHANGES INTO LIVE.customers_silver
  FROM STREAM(LIVE.customers_bronze_clean)
  KEYS (customer_id)
  APPLY AS DELETE WHEN operation = "DELETE"
  SEQUENCE BY timestamp
  COLUMNS * EXCEPT (operation, _rescued_data)